In [ ]:
import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter

DATA_DIR = Path('../data/processed/waymo_e2e/val')
LABEL_JSON = Path('../data/val_sequence_name_to_scenario_cluster.json')

print(f'Data dir exists: {DATA_DIR.exists()}')
print(f'Label JSON exists: {LABEL_JSON.exists()}')

## Find Class Label data

In [ ]:
LABEL_JSON

In [ ]:
with open(LABEL_JSON) as f:
    labels = json.load(f)

In [ ]:
# Peek at the JSON structure
first_key = list(labels.keys())[0]
print(f'Key: {first_key}')
print(f'Value: {labels[first_key]}')

## Create sequence dictionary and organize npz files
- use Dawsons load_image function

In [ ]:
npz_files = sorted(DATA_DIR.glob('**/*.npz'))

In [ ]:
# Dictionary to store each unique video/sequence and its frame numbers
seq_to_frames = {}
bad_files = []

print("total npz files:", len(npz_files))

# Find unique videos/sequences
for filepath in npz_files:
    
    # Get just the file name without the .npz extension
    filename = filepath.stem

    # Split on the underscore
    sequence_id, frame_id_text = filename.rsplit("_", 1)

    # Make sure the frame id is actually numeric
    # This skips files like: 0075..._91(1).npz
    if not frame_id_text.isdigit():
        bad_files.append(filepath)
        continue
    
    # Convert frame_id from string to integer
    frame_id = int(frame_id_text)

    # If this sequence_id has not been seen yet, create an empty list for it
    if sequence_id not in seq_to_frames:
        seq_to_frames[sequence_id] = []

    # Add the frame number to that sequence's frame list
    seq_to_frames[sequence_id].append(frame_id)

# Sort the frame IDs for each sequence
for sequence_id in seq_to_frames:
    seq_to_frames[sequence_id] = sorted(seq_to_frames[sequence_id])

print("unique videos/sequences:", len(seq_to_frames))
print("bad/skipped files:", len(bad_files))

In [ ]:
def load_image(fpath):
    data = np.load(fpath, allow_pickle=True)
    modality = data['_modality_data'].item()
    keys = list(modality.keys())
    img = modality[keys[0]]  # CAMERAS is always first key
    if img.dtype in [np.float32, np.float64]:
        img = np.clip(img, 0, 1)
    else:
        img = img.astype(np.uint8)
    return img

## Function to classify day vs night
- Use only the first frame of each sequence to save computation
- Threshold value is tunable.
- brightness score is between 0 and 1

In [ ]:
def classify_day_night_from_image(img, threshold=0.30):
    """
    Classify one image as day or night using average brightness.

    Assumes img is an RGB image with shape:
        height x width x 3

    Returns:
        label: 'day' or 'night'
        brightness_score: average brightness from 0 to 1
    """

    img = np.asarray(img)

    # Convert image to float between 0 and 1
    if img.dtype == np.uint8:
        img = img.astype(np.float32) / 255.0
    else:
        img = img.astype(np.float32)

        # If image values are 0 to 255 but stored as float, normalize them
        if img.max() > 1.0:
            img = img / 255.0

    # Remove alpha channel if present
    if img.ndim == 3 and img.shape[2] > 3:
        img = img[:, :, :3]

    # Use the upper half of the image because sky/lighting is usually more useful
    # than the road for deciding day vs night.
    upper_half = img[:img.shape[0] // 2, :, :3]

    # Convert RGB to perceived brightness/luminance
    luminance = (
        0.299 * upper_half[:, :, 0] +
        0.587 * upper_half[:, :, 1] +
        0.114 * upper_half[:, :, 2]
    )

    brightness_score = luminance.mean()

    if brightness_score >= threshold:
        label = "day"
    else:
        label = "night"

    return label, brightness_score


def classify_first_frame_day_night(seq_to_frames, npz_files, threshold=0.30):
    """
    Looks at the first frame in each sequence and classifies the sequence
    as day or night.

    Uses:
        seq_to_frames: dictionary where sequence_id -> sorted list of frame ids
        npz_files: list of .npz file paths
        load_image function

    Returns:
        DataFrame with one row per sequence
    """

    # Build lookup once:
    # (sequence_id, frame_id) -> filepath
    frame_lookup = {}
    bad_files = []

    for filepath in npz_files:
        filename = filepath.stem

        try:
            seq_id, frame_id_text = filename.rsplit("_", 1)
        except ValueError:
            bad_files.append(filepath)
            continue

        # Skip malformed filenames like ..._91(1).npz
        if not frame_id_text.isdigit():
            bad_files.append(filepath)
            continue

        frame_id = int(frame_id_text)
        frame_lookup[(seq_id, frame_id)] = filepath

    results = []
    missing_sequences = []

    for sequence_id, frame_ids in seq_to_frames.items():

        # Get first frame in the sequence
        first_frame_id = sorted(frame_ids)[0]

        key = (sequence_id, first_frame_id)

        if key not in frame_lookup:
            missing_sequences.append(sequence_id)
            continue

        fpath = frame_lookup[key]

        # Load first image from this sequence
        img = load_image(fpath)

        # Classify as day or night
        label, brightness_score = classify_day_night_from_image(
            img,
            threshold=threshold
        )

        results.append({
            "sequence_id": sequence_id,
            "first_frame_id": first_frame_id,
            "day_night": label,
            "brightness_score": brightness_score,
            "filepath": fpath
        })

    results_df = pd.DataFrame(results)

    print("Sequences classified:", len(results_df))
    print("Missing sequences:", len(missing_sequences))
    print("Bad files skipped:", len(bad_files))

    return results_df

In [ ]:
day_night_df = classify_first_frame_day_night(
    seq_to_frames=seq_to_frames,
    npz_files=npz_files,
    threshold=0.30
)

day_night_df.head()

In [ ]:
day_night_df.describe()

In [ ]:
day_night_df['day_night'].value_counts()

In [ ]:
day_only_df = day_night_df[day_night_df['day_night'] == 'day']
day_only_df

## Visualize an image with the lowest brightness score
- This will help us validate our chosen threshold

In [ ]:
lowest_brightness = day_only_df['brightness_score'].min()
lowest_brightness

In [ ]:
# Find the row index with the lowest brightness score
lowest_idx = day_only_df["brightness_score"].idxmin()

# Pull out the full row
lowest_row = day_only_df.loc[lowest_idx]

lowest_row

In [ ]:
sequence_id = lowest_row["sequence_id"]
frame_id = lowest_row["first_frame_id"]

npz_filename = f"{sequence_id}_{frame_id}.npz"

npz_filename

In [ ]:
lowest_npz_path = None

for filepath in npz_files:
    if filepath.name == npz_filename:
        lowest_npz_path = filepath
        break

lowest_npz_path

In [ ]:
img = load_image(lowest_npz_path)
print(f'Image shape: {img.shape}, dtype: {img.dtype}')
plt.figure(figsize=(14, 3))
plt.imshow(img)
plt.axis('off')
plt.title(f'Lowest brightness from filtered dataframe — {lowest_npz_path.name}')
plt.tight_layout()
plt.show()

NOTE threshold at 0.3 may be too high. We can play with the tuning later. The lowest brightness score here seems to be affected by the overhead bridge...

## Add all frames to the day only dataframe
- add class labels

In [ ]:
def expand_day_only_to_all_frames(day_only_df, seq_to_frames, npz_files, labels):
    """
    Expands day_only_df from one row per sequence to one row per frame.

    Inputs:
        day_only_df:
            DataFrame containing only daytime sequences.
            Must contain a 'sequence_id' column.

        seq_to_frames:
            Dictionary where:
                sequence_id -> sorted list of frame ids

        npz_files:
            List of .npz file paths.

        labels:
            Dictionary loaded from val_sequence_name_to_scenario_cluster.json.
            Key is sequence_id.
            Value contains scenario_cluster.

    Returns:
        DataFrame with one row per frame from daytime sequences.
    """

    # Build lookup:
    # (sequence_id, frame_id) -> filepath
    frame_lookup = {}
    bad_files = []

    for filepath in npz_files:
        filename = filepath.stem

        try:
            seq_id, frame_id_text = filename.rsplit("_", 1)
        except ValueError:
            bad_files.append(filepath)
            continue

        # Skip malformed duplicate files like ..._91(1).npz
        if not frame_id_text.isdigit():
            bad_files.append(filepath)
            continue

        frame_id = int(frame_id_text)
        frame_lookup[(seq_id, frame_id)] = filepath

    rows = []

    # Only expand the sequence IDs that are in day_only_df
    day_sequence_ids = day_only_df["sequence_id"].unique()

    for sequence_id in day_sequence_ids:

        # Skip if this sequence is not in seq_to_frames
        if sequence_id not in seq_to_frames:
            continue

        # Get class label from JSON
        class_label = labels.get(sequence_id, {}).get("scenario_cluster", None)

        # Optional cleanup of label typos/inconsistent formatting
        if class_label == "Interections":
            class_label = "Intersections"
        elif class_label == "Cut_ins":
            class_label = "Cut-ins"

        # Get the original day/night info for this sequence
        sequence_row = day_only_df[day_only_df["sequence_id"] == sequence_id].iloc[0]

        for frame_id in sorted(seq_to_frames[sequence_id]):

            key = (sequence_id, frame_id)

            if key in frame_lookup:
                filepath = frame_lookup[key]
            else:
                filepath = None

            rows.append({
                "sequence_id": sequence_id,
                "frame_id": frame_id,
                "npz_filename": f"{sequence_id}_{frame_id}.npz",
                "filepath": filepath,
                "day_night": sequence_row["day_night"],
                "first_frame_id": sequence_row["first_frame_id"],
                "first_frame_brightness_score": sequence_row["brightness_score"],
                "scenario_cluster": class_label
            })

    expanded_df = pd.DataFrame(rows)

    print("Original day-only sequences:", len(day_sequence_ids))
    print("Expanded frame rows:", len(expanded_df))
    print("Bad/skipped npz files:", len(bad_files))

    return expanded_df

In [ ]:
day_only_frames_df = expand_day_only_to_all_frames(
    day_only_df=day_only_df,
    seq_to_frames=seq_to_frames,
    npz_files=npz_files,
    labels=labels
)

day_only_frames_df

In [ ]:
day_only_frames_df["scenario_cluster"].value_counts()

In [ ]:
day_only_frames_df.groupby("scenario_cluster")["sequence_id"].nunique()

Next filtering step is to choose three classes - (cyclists, pedestrians, ???) We should choose a third class that exists in YOLO. Then use Yolo to determine which frame in each sequence is the best one to use for a given class

## Save filtered day time images (all frames) as pngs to filtered folder
- save dataframe with labels as csv

In [ ]:
import imageio.v2 as imageio

def save_day_only_frames_as_png(day_only_frames_df, output_dir="../data/filtered_dayOnly"):
    """
    Save the raw image for every row in day_only_frames_df as a PNG.

    Assumes day_only_frames_df contains:
        - sequence_id
        - frame_id
        - filepath

    Assumes load_image(filepath) is already defined.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved_count = 0
    skipped_count = 0

    for _, row in day_only_frames_df.iterrows():
        sequence_id = row["sequence_id"]
        frame_id = row["frame_id"]
        fpath = row["filepath"]

        # Skip rows with missing filepath
        if fpath is None:
            skipped_count += 1
            continue

        # Load raw image from the npz file
        img = load_image(fpath)
        img = np.asarray(img)

        # If image is float in range 0 to 1, convert to uint8
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            else:
                img = img.astype(np.uint8)

        # Remove alpha channel if present
        if img.ndim == 3 and img.shape[2] > 3:
            img = img[:, :, :3]

        # Output filename
        out_name = f"{sequence_id}_{frame_id}.png"
        out_path = output_dir / out_name

        # Save raw image directly - no matplotlib border
        imageio.imwrite(out_path, img)

        saved_count += 1

    print("Done.")
    print("Saved images:", saved_count)
    print("Skipped rows:", skipped_count)
    print("Output folder:", output_dir.resolve())

In [ ]:
save_day_only_frames_as_png(day_only_frames_df, output_dir="../data/filtered_dayOnly")

In [ ]:
# Set output folder
output_dir = Path("../data")

# Save dataframe as CSV
out_path = output_dir / "day_only_frames_df.csv"

day_only_frames_df.to_csv(out_path, index=False)

print("Saved dataframe to:")
print(out_path.resolve())